# Painter (diffusion) — one-click Colab training

Train the morphengine **painter** (before → after edit) on a free Colab GPU:
SDXL base + LoRA adapters with geometry conditioning, driven entirely by the
repo's own code (`morphengine.painter.train`).

**Prerequisites:** a Google account and a GPU runtime
(*Runtime → Change runtime type → T4 GPU*). The defaults below are a
T4-sized demo run (~256 px, batch 4, 2000 steps ≈ 30–60 min); the full
`configs/painter_v0.yaml` (512 px, batch 8, 20k steps) targets an A100 —
see `src/morphengine/painter/README.md`.

## 1. Setup — clone the repo, install deps, check the GPU

Clones `main` of the public repo to `/content/ps` and installs the
`painter` extra (`torch diffusers peft accelerate transformers`, per
`pyproject.toml`). Dirs from other workstreams (`service/`, `notebooks/`)
may be absent on `main` — nothing here depends on them.

In [ ]:
# idempotent clone: re-running this cell (or Run-all on a reused runtime)
# must not die on "destination path exists"
from pathlib import Path as _P
if (_P("/content/ps") / ".git").exists():
    !git -C /content/ps fetch origin main && git -C /content/ps reset --hard origin/main
elif _P("/content/ps").exists():
    !rm -rf /content/ps  # partial/broken clone from an earlier run
    !git clone https://github.com/bustamonica/plasticsurgery.git /content/ps && git -C /content/ps checkout main
else:
    !git clone https://github.com/bustamonica/plasticsurgery.git /content/ps && git -C /content/ps checkout main
%cd /content/ps
# CUDA 12.1 torch wheels first, so the [painter] extra reuses them:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -e "/content/ps[painter]"
!pip install -q "diffusers>=0.29" "transformers>=4.40" "accelerate>=0.30" "peft>=0.11" "safetensors>=0.4"

import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
import morphengine
print("morphengine at:", morphengine.__file__)

## 2. Dataset — upload the demo zip *or* generate a small set

Training consumes a datafactory `manifest.jsonl` (before/after PNGs +
depth/normal/mask `.npy`). Rendering ~1500 pairs on Colab's 2 CPU cores is
too slow, so the default path is: **no upload needed** — `m2_dataset_demo.zip` (23 real-body pairs)
ships in this repo under `notebooks/` and is found automatically after the
clone. (You can still upload a zip manually to override.) via the file panel, then
run the cell. Set `GENERATE = True` to render 300 fresh pairs instead
(slower, but fully deterministic).

In [ ]:
GENERATE = False  # False: unzip the repo-shipped m2 demo zip (or an uploaded override); True: render 300 pairs on the fly

from pathlib import Path
import json

DATA_ROOT = Path("/content/data")

if GENERATE:
    # scripts/generate_dataset.py CLI: --n --seed --size --out --resolution --verbose
    !python /content/ps/scripts/generate_dataset.py --n 300 --seed 0 --size 256 --out /content/data/painter_v0 --verbose
    MANIFEST = "/content/data/painter_v0/manifest.jsonl"
else:
    # Upload m1_dataset_demo.zip first (Files panel, or uncomment):
    # from google.colab import files; files.upload()
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    import glob
    zips = sorted(glob.glob('/content/ps/notebooks/m*_dataset_demo.zip') + glob.glob('/content/m*_dataset_demo.zip'))
    hits = sorted(DATA_ROOT.rglob("manifest.jsonl"))  # tolerate any zip layout
    assert hits, "no manifest.jsonl under /content/data — did the zip upload?"
    MANIFEST = str(hits[0])

rows = [json.loads(l) for l in open(MANIFEST) if l.strip()]
print("MANIFEST =", MANIFEST)
print(f"{len(rows)} pairs")

## 3. Config — load `configs/painter_v0.yaml`

`train.py` has no config-file loader or CLI (no `__main__`); the runbook
(`painter/README.md` §4) maps the yaml keys onto `TrainConfig` explicitly,
and this cell does the same. The yaml's extra keys
(`lr_schedule`, `optimizer`, `precision`, `ema_decay`, `grad_clip`,
`log_every`, `save_every`, `data_manifest`) are **not** consumed by
`TrainConfig` — `TrainConfig(**yaml)` would raise, so don't splat it.
Edit `OVERRIDES` for quick experiments.

In [ ]:
import yaml
from morphengine.painter.train import TrainConfig

cfg_dict = yaml.safe_load(open("/content/ps/configs/painter_v0.yaml"))

# T4-friendly overrides (the raw yaml targets an A100 — README §5):
OVERRIDES = {
    "image_size": 256,  # yaml: 512
    "batch_size": 4,    # yaml: 8
    "steps": 2000,      # yaml: 20000 — demo run; raise for real weights
    # "lr": 1.0e-4,
}
cfg_dict.update(OVERRIDES)

cfg = TrainConfig(
    model="sdxl-lora",
    image_size=cfg_dict["image_size"],
    lr=cfg_dict["lr"],
    batch_size=cfg_dict["batch_size"],
    steps=cfg_dict["steps"],
    lora_rank=cfg_dict["lora_rank"],
    lora_alpha=cfg_dict["lora_alpha"],
    base_model=cfg_dict["base_model"],
    out_dir="/content/ckpt",  # checkpoints land here for the export step
    seed=cfg_dict["seed"],
)
print("Resolved TrainConfig:")
for k, v in vars(cfg).items():
    print(f"  {k}: {v}")

## 4. Train — SDXL + LoRA via the repo entrypoint

`morphengine.painter.train.train(cfg, manifest)` is the only entrypoint
(library-callable; `python -m morphengine.painter.train` does nothing —
no argparse). What it does: frozen SDXL VAE, UNet `conv_in` extended
4→8 ch for geometry conditioning, LoRA (rank/alpha from config) on
`to_q/to_k/to_v/to_out.0`, epsilon-MSE + AdamW + cosine LR, EMA export.

**Known gaps in the current `train.py` sdxl-lora path** (run what it does;
don't expect prompt-following or mid-run checkpoints):

- **Prompt conditioning is a placeholder** — zero text embeds, the model
  is effectively unconditional (`train.py` lines ~264–268, ~317–326).
- **`save_every` is ignored** — no periodic checkpointing; weights are
  written once at the end into `out_dir` (`/content/ckpt`). Keep
  `steps` small enough to finish inside the Colab session.
- **`grad_clip` / `precision` yaml keys are not read**; bf16 is chosen
  automatically when CUDA is present, and there is no grad clipping.
- **Suspected bug (reported, not patched here):** the extended `conv_in`
  is installed *before* `get_peft_model` (`train.py` ~221–249), and peft
  freezes all non-adapter params — so the new conditioning channels may
  stay frozen at zero-init, i.e. geometry conditioning currently has no
  gradient. LoRA itself still trains.

In [ ]:
import time
from morphengine.painter.train import train

t0 = time.time()
result = train(cfg, manifest=MANIFEST)
print(f"done in {(time.time() - t0) / 60:.1f} min")
result  # {"final_loss": ..., "steps": ..., "ckpt": "/content/ckpt/checkpoint_last.pt"}

## 5. Sample — conditioning → painter output (2×4 grid)

The inference entrypoint lives in the repo:
`morphengine.painter.inference.PainterInference`. `from_ckpt` rebuilds the
exact training-time graph from the export (base SDXL UNet → 4→8 ch
`conv_in` extension → `unet_lora` peft adapter → `cond_encoder` from
`cond_encoder.pt`, DDPM scheduler, zero prompt embeds — unconditional;
geometry, not text, steers the edit). `paint(before, cond, steps, seed)`
runs the sampler; the 6-channel cond convention is shared with training via
`painter.dataset.build_cond` (single source of truth — never rebuilt inline).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from morphengine.painter.dataset import PairDataset
from morphengine.painter.inference import PainterInference

painter = PainterInference.from_ckpt("/content/ckpt")  # reads config.json for base model + image size
ds = PairDataset(MANIFEST, image_size=painter.image_size)

idxs = np.linspace(0, len(ds) - 1, 4).astype(int)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for j, i in enumerate(idxs):
    item = ds[int(i)]
    out_img = painter.paint(item["before"].numpy(), item["cond"].numpy(),
                            steps=30, seed=int(i))  # seeded -> reproducible samples
    axes[0, j].imshow(item["before"].numpy().transpose(1, 2, 0) * 0.5 + 0.5)
    axes[0, j].set_title(f"before #{i}")
    axes[1, j].imshow(out_img)
    axes[1, j].set_title("painter output")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Export — zip the checkpoints and download

`/content/ckpt/` holds `unet_lora/` (peft adapter, EMA weights),
`cond_encoder.pt`, `checkpoint_last.pt`, and `config.json` — everything
M2 inference needs (see `painter/README.md` §7). The zip downloads via
your browser; nothing is persisted when the Colab session ends.

In [ ]:
!cd /content && zip -qr painter_ckpt.zip ckpt && ls -lh painter_ckpt.zip
from google.colab import files
files.download("/content/painter_ckpt.zip")